# Playbook: Hugging Face Tour

> **Курс «От нуля до своих агентов» — Модуль 1.5.**
> Демо-сценарий, по которому я провожу живую экскурсию по HF.
> Часть времени — переключаемся в браузер и смотрим UI Hub,
> часть — запускаем код прямо здесь, в этом ноутбуке.
>
> Длительность: ~20 минут.
> Что в конце: у каждого — токен HF в Kaggle Secrets, запущенный
> `pipeline("sentiment-analysis")`, загруженный датасет IMDB, понимание,
> что такое model card и Space.

> **Предпосылка:** запускать этот ноутбук имеет смысл уже в Kaggle —
> чтобы переиспользовать настроенную среду из `kaggle-tour.ipynb`.

## 0. Зачем нам Hugging Face

Простой ответ: это GitHub для весов моделей и датасетов. Без него
работа с открытыми моделями превращается в «скачайте zip-архив с
сайта университета 2019 года».

Длинный ответ: HF Hub даёт три вещи:

1. **Готовые модели** — миллион+ моделей с открытыми весами. Каждая —
   как репозиторий с карточкой README, файлами весов и примерами.
2. **Готовые датасеты** — тысячи стандартизованных датасетов, которые
   подключаются строкой кода.
3. **Spaces** — бесплатный хостинг для демо-приложений на Gradio или
   Streamlit. На Spaces поднимают тысячи live-демо моделей.

Плюс библиотеки `transformers`, `datasets`, `huggingface_hub` —
которыми всё это используется из кода.

## 1. Регистрация и токен (web)

1. Открыть https://huggingface.co/join, зарегистрироваться.
2. Подтвердить почту.
3. Профиль (правый верхний угол) → **Settings** → **Access Tokens**.
4. **New token** → Type = **Read** (Write нужен только когда вы сами
   публикуете модель).
5. Имя: `train-of-thought-course`.
6. **Скопировать токен сразу** — второй раз он не покажется.

**Положить токен в Kaggle Secrets:**
`Add-ons` → `Secrets` → `Add a new secret` → имя `HF_TOKEN`, значение —
сам токен. Галка **Attach to notebook** обязательна.

Достаём в коде:

In [ ]:
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
print("HF токен загружен, длина:", len(HF_TOKEN), "символов")

## 2. Экскурсия по Hub (web — переключаемся в браузер)

Открываем https://huggingface.co и идём по верхнему меню:

- **Models** — фильтры слева: задача (Text Generation, Image
  Classification...), язык, лицензия, размер. Покажу:
  - `meta-llama/Llama-3.1-8B-Instruct` — образцовая open-weights LLM.
  - `sentence-transformers/all-MiniLM-L6-v2` — крошечная (~25M)
    embedding-модель, дефолт для семантического поиска.
  - `distilbert-base-uncased-finetuned-sst-2-english` — классификатор
    настроения, на котором мы скоро запустим `pipeline`.
- **Datasets** — фильтры по задаче и языку. Покажу:
  - `imdb` — 50k отзывов, бинарная разметка POSITIVE/NEGATIVE.
  - `c4` — гигантский веб-корпус, на котором обучали T5.
- **Spaces** — открываем 2-3 демо: `gradio/hello_world`, любой
  свежий vision-демо.

**Что в каждой странице важно показать:**

- слева **Files** — здесь живут сами веса (обычно `.safetensors` или
  `.bin`);
- сверху **Use this model** — кнопка с готовым сниппетом кода;
- внизу карточка с **license**, **datasets used**, **community**.

## 3. Анатомия model card

На примере одной модели разбираем, что читать **перед** тем как взять её
в работу. Идём на:
https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Три блока, которые надо прочесть всегда:

1. **License** — указана сверху и сбоку. `apache-2.0`, `mit` —
   используете коммерчески спокойно. `llama3`, `gemma` — есть
   оговорки. `cc-by-nc` — некоммерчески. Просто «опубликовано на HF»
   не значит «можно где угодно».
2. **Intended uses / Out-of-scope use** — часто авторы сами пишут
   «работает только на английском», «не для медицинских решений», и
   так далее.
3. **Model size / Inference requirements** — сколько ГБ памяти ест.
   На Kaggle T4 (15 ГБ VRAM) модель в 7B параметров с fp16 поместится
   впритык. 70B — не поместится никак.

**Правило большого пальца для размера в памяти:**

- fp32 (полная точность): N миллиардов параметров × 4 ГБ
- fp16/bf16: N × 2 ГБ
- int8 (квантизация): N × 1 ГБ
- int4: N × 0.5 ГБ

Llama-3.1-8B → 16 ГБ в fp16 → впритык на T4 → имеет смысл взять
int4-квантизованную версию.

## 4. Первая модель за 3 строки — pipeline

Самое простое API HF — `pipeline`. Под капотом он:

1. Идёт на Hub, скачивает дефолтную модель для указанной задачи.
2. Кладёт её в локальный кэш (`~/.cache/huggingface/` или `HF_HOME`).
3. Поднимает токенизатор + модель + пост-обработку результата.

Три строки — и вы делаете то, что в 2018-м стоило недели работы.

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline

# device=-1 — принудительно CPU. На Kaggle бывает старый GPU (Tesla
# P100), который несовместим со свежим PyTorch. Для крошечной distilbert
# CPU быстрее по запуску и надёжнее.
clf = pipeline("sentiment-analysis", device=-1)  # дефолт: distilbert SST-2, ~250 МБ
print(clf("This course is unexpectedly fun."))
print(clf("Honestly, I'm just here for the certificate."))
print(clf("I had high hopes but the second module was a slog."))

**Что произошло:**

- `pipeline("sentiment-analysis")` без аргументов взял дефолтную модель
  для этой задачи — `distilbert-base-uncased-finetuned-sst-2-english`.
  Это исторический классификатор на SST-2 датасете.
- Вернулся список словарей вида `{'label': 'POSITIVE', 'score': 0.99}`.
- В скоре — уверенность модели, не вероятность правды.

**Если хотите другую модель** — передайте её id в `model=`:

In [ ]:
# Многоязычная sentiment-модель — работает на русском, английском, испанском, и т.д.
multi = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
)
print(multi("Этот курс мне очень нравится!"))
print(multi("Не понимаю, что тут происходит."))

## 5. Датасеты за 2 строки — load_dataset

Та же история с датасетами. Стандартизованный API, кэширование, не
надо качать архивы.

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

# Первые 100 примеров из test-сплита IMDB
ds = load_dataset("imdb", split="test[:100]")
print("Тип:", type(ds).__name__)
print("Размер:", len(ds))
print("Поля:", ds.column_names)
print("Пример:")
print(" текст:", ds[0]["text"][:200], "...")
print(" метка:", ds[0]["label"], "(0 = негативный, 1 = позитивный)")

**Считаем accuracy на лету:** прогоним sentiment-классификатор
на этих 100 отзывах и сравним с настоящими метками.

In [ ]:
correct = 0
for row in ds:
    pred = clf(row["text"][:512])[0]["label"]  # обрезаем, чтобы влезло в 512 токенов
    true_label = "POSITIVE" if row["label"] == 1 else "NEGATIVE"
    if pred == true_label:
        correct += 1

print(f"Accuracy на 100 отзывах IMDB: {correct}/{len(ds)} = {correct/len(ds):.0%}")

**Что важно заметить:**

- Эта модель училась на SST-2 (короткие фразы), а IMDB — длинные отзывы.
  Поэтому accuracy не идеальная. Это нормальная история «модель из
  одной задачи на соседней работает хуже».
- Обрезка до 512 символов грубая. В проде надо токенизировать и считать
  длину в токенах. Об этом — в Модуле 5.5.

## 6. Кэш и место на диске

Каждый вызов `pipeline(...)` или `load_dataset(...)` качает данные в
кэш `~/.cache/huggingface/`. На Kaggle это часть рабочей папки, лимит
порядка 20 ГБ.

Если планируете жонглировать большими моделями — переместите кэш во
временную папку Kaggle, у которой больше места:

In [ ]:
import os

os.environ["HF_HOME"] = "/kaggle/temp/hf"  # /kaggle/temp — больше места
# После смены HF_HOME перезапустите ядро ноутбука: Run → Restart.
print("HF_HOME =", os.environ["HF_HOME"])

In [ ]:
# Посмотреть, что сейчас в кэше:
!du -sh ~/.cache/huggingface 2>/dev/null || echo "Кэш пуст или папки нет"

## 7. Spaces — куда выкладывать своё демо

Spaces — это бесплатный хостинг для веб-приложений с моделями. Под
капотом — Docker-контейнер с Gradio или Streamlit. Подходит, чтобы:

- показать друзьям работу своей модели;
- сделать публичное демо к ДЗ или к pet-project'у;
- быстро попробовать чужую модель прямо в браузере.

Создание Space:

1. На сайте HF: **+ New Space**.
2. SDK: **Gradio** (самое простое) или **Streamlit**.
3. Hardware: **CPU basic** бесплатный.
4. Создать → у вас репозиторий с шаблоном `app.py`.

Самый короткий `app.py` на Gradio:

```python
import gradio as gr
from transformers import pipeline

clf = pipeline("sentiment-analysis")

def classify(text):
    result = clf(text)[0]
    return f"{result['label']} (score={result['score']:.2f})"

gr.Interface(fn=classify, inputs="text", outputs="text").launch()
```

Push в репозиторий → Space сам пересоберётся и поднимет публичный
URL. Подробнее будет в Модуле 16 (свой агент), когда придёт время
показать миру работу своего bota.

## 8. Что должно получиться у каждого

К концу демонстрации:

- [ ] Аккаунт HF создан, токен `Read` выпущен.
- [ ] Токен лежит в Kaggle Secrets как `HF_TOKEN`.
- [ ] `pipeline("sentiment-analysis")` запустился, ответ на свою фразу
      пришёл.
- [ ] `load_dataset("imdb")` отработал, видно поля и пример.
- [ ] Открыли хотя бы одну карточку модели на HF и прочитали лицензию.

Следующий шаг — играем по этой инфраструктуре в основное ДЗ Модуля 1.5
(см. лекцию: «Hello-ноутбук со всеми тремя площадками»).